In [ ]:
import pandas as pd
import sys
from pathlib import Path
import asyncio
import time
import gc

sys.path.append(str(Path().resolve().parent.parent))
from src.gradient_client import GradientSportsClient

pd.set_option('display.max_columns', None)

### Status da requisição

In [2]:
client = GradientSportsClient()

# health check
client.get_status()

{'data': {'status': 'ok'}}

In [3]:
games = pd.read_csv(str(Path().resolve().parent.parent / "data" / "games.csv"))

# remover jogos da temporada atual pois não serão usados
games = games[~games['season'].isin(['2025-2026', '2026'])].reset_index(drop=True)
games

,id,date,season,teamExtraTimeStartSide,teamStartSide,venueType,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width
0,4447,2022-08-13,2022-2023,Right,Left,TEAM_HOME,3,Aston Villa,1,Premier League,8,Everton,Villa Park,105.0,68.0
1,4760,2023-04-25,2022-2023,Left,Left,OPPONENT_HOME,7,Crystal Palace,1,Premier League,20,Wolverhampton Wanderers,Molineux,105.0,68.0
2,12806,2023-11-04,2023,Left,Right,OPPONENT_HOME,517,Bahia,42,Brasileiro Série A,515,Grêmio,Arena do Grêmio,105.0,68.0
3,32206,2025-01-18,2024-2025,Right,Left,TEAM_HOME,119,Brentford,1,Premier League,10,Liverpool,Gtech Community Stadium,105.0,68.0
4,4451,2022-08-15,2022-2023,Left,Left,OPPONENT_HOME,7,Crystal Palace,1,Premier League,10,Liverpool,Anfield,101.0,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3021,231,2020-11-21,2020-2021,Right,Left,TEAM_HOME,3,Aston Villa,1,Premier League,4,Brighton & Hove Albion,Villa Park,105.0,68.0
3022,4660,2023-02-12,2022-2023,Right,Left,TEAM_HOME,55,Leeds United,1,Premier League,12,Manchester United,Elland Road,105.0,68.0
3023,1224,2021-12-04,2021-2022,Left,Left,OPPONENT_HOME,10,Liverpool,1,Premier League,20,Wolverhampton Wanderers,Molineux,105.0,68.0
3024,12744,2023-09-30,2023,Right,Right,TEAM_HOME,437,Fortaleza,42,Brasileiro Série A,515,Grêmio,Arena Castelao,105.0,68.0


In [ ]:
SEM_LIMIT = 5
semaphore = asyncio.Semaphore(SEM_LIMIT)

# Cria a pasta "events" caso não exista
events_dir = Path().resolve().parent.parent / "data" / "events"
events_dir.mkdir(parents=True, exist_ok=True)  # Cria a pasta se não existir

async def fetch_game_events(game, index, total):
    game_id = game['id']
    game_season = game['season']
    game_competition = game['competition.id']
    
    async with semaphore:
        print(f"[{index + 1} / {total}] Starting request for game: {game_id}, competition id: {game_competition}, season: {game_season}")
        start_time = time.perf_counter()
        
        await asyncio.sleep(0.2)  # Controle para não estourar API
        
        df = await asyncio.to_thread(
            client.get_game_events_flat,
            game_id,
            as_dataframe=True
        )
        
        elapsed = time.perf_counter() - start_time
        print(f"[{index + 1} / {total}] Finished request for game {game_id} in {elapsed:.2f} seconds")
    
    df['competitionId'] = game_competition
    df['gameId'] = game_id
    df['season'] = game_season
    return df, game_competition, game_id, game_season

async def process_group(group_df):
    games_list = [row for _, row in group_df.iterrows()]
    total = len(games_list)
    
    tasks = [
        asyncio.create_task(fetch_game_events(game, index, total))
        for index, game in enumerate(games_list)
    ]
    
    for task in asyncio.as_completed(tasks):
        df, competition_id, game_id, season = await task
        
        events_dir = Path().resolve().parent.parent / "data" / "events"/ f"{competition_id}" / f"{season}"
        events_dir.mkdir(parents=True, exist_ok=True)  # Cria a pasta se não existir

        file_name = f"events_{game_id}.parquet"
        file_path = events_dir / file_name
        df.to_parquet(file_path, index=False)
        
        print(f"Saved {file_name} with {len(df)} rows")
        
        # Limpar memória antes de próxima iteração (se necessário)
        del df
        gc.collect()

async def main():
    # Garanta que a coluna 'competition.id' e 'season' existam no DataFrame games
    grouped = games.groupby(['competition.id', 'season'])
    
    for (competition, season), group_df in grouped:
        print(f"\nProcessing competition id {competition}, season {season} - {len(group_df)} games")
        
        await process_group(group_df)
        
        print(f"Process completed for games of competition id {competition} and season {season}")

await main()


Processing competition id 1, season 2020-2021 - 378 games
[1 / 378] Starting request for game: 659, competition id: 1, season: 2020-2021
[2 / 378] Starting request for game: 287, competition id: 1, season: 2020-2021
[3 / 378] Starting request for game: 391, competition id: 1, season: 2020-2021
[4 / 378] Starting request for game: 414, competition id: 1, season: 2020-2021
[5 / 378] Starting request for game: 386, competition id: 1, season: 2020-2021
[4 / 378] Finished request for game 414 in 10.86 seconds
[6 / 378] Starting request for game: 426, competition id: 1, season: 2020-2021
Saved events_414.parquet with 2875 rows
[5 / 378] Finished request for game 386 in 13.15 seconds
[7 / 378] Starting request for game: 472, competition id: 1, season: 2020-2021
Saved events_386.parquet with 2549 rows
[3 / 378] Finished request for game 391 in 14.97 seconds
[8 / 378] Starting request for game: 164, competition id: 1, season: 2020-2021
Saved events_391.parquet with 2645 rows
[1 / 378] Finished

CancelledError: 

[23 / 378] Finished request for game 381 in 9.20 seconds
[28 / 378] Starting request for game: 332, competition id: 1, season: 2020-2021
[26 / 378] Finished request for game 466 in 6.61 seconds
[29 / 378] Starting request for game: 329, competition id: 1, season: 2020-2021
[25 / 378] Finished request for game 179 in 13.53 seconds
[30 / 378] Starting request for game: 369, competition id: 1, season: 2020-2021
[27 / 378] Finished request for game 367 in 12.96 seconds
[31 / 378] Starting request for game: 435, competition id: 1, season: 2020-2021
[24 / 378] Finished request for game 323 in 17.72 seconds
[32 / 378] Starting request for game: 324, competition id: 1, season: 2020-2021
[28 / 378] Finished request for game 332 in 13.39 seconds
[33 / 378] Starting request for game: 492, competition id: 1, season: 2020-2021
[29 / 378] Finished request for game 329 in 10.04 seconds
[34 / 378] Starting request for game: 352, competition id: 1, season: 2020-2021
[30 / 378] Finished request for game